In [1]:
import numpy as np
# import get_site_interactions
from matplotlib.path import Path
from scipy.signal import medfilt
from scipy.io import loadmat, savemat

In [2]:
''' Set root directory '''
# root_dir = "Z:/Isabel/data/hpc_implants/" # locker
root_dir = "C:/Users/ilow1/Documents/code/bird_pose_tracking/model_output/" # local - update as needed
bird_id = 'RBY94'
session_id = '241129'
session_root = f"{root_dir}{bird_id}/{bird_id}_{session_id}/"


''' Define paths to models, smoothed keypoints, arena info '''
pred_file = f'250109_posture_2stage_face.npy'
pred_path = f"{session_root}{pred_file}"

behavior_folder = f"{session_root}/behavior_data/"
pos_file = 'posture_pos_smooth.npy'
vel_file = 'posture_vel_smooth.npy'

arena_dir = 'C:/Users/ilow1/Documents/code/il_rig_control/arena_alignment/'
arena_items_file = 'arena_items_2.mat'

In [3]:
''' load the smoothed keypoints and calculate the foot speed '''
print('\nLoading smoothed postural keypoints and velocity...')
smooth_pts = np.load(f"{behavior_folder}{pos_file}") # n_frames, n_keypoints, 3
smooth_vel = np.load(f"{behavior_folder}{vel_file}")
foot_speed = np.sqrt(np.sum(np.mean(smooth_vel[:, [10, 14]], axis=1)**2, axis=1))

''' load the model output and get the body reprojection error '''
print('\nGetting reprojection error...')
results_dict = np.load(pred_path, allow_pickle=True).item()
results = results_dict['results']
body_reproj_error = results['com_rep_err'][:, 1]

''' load the arena objects '''
print('\nGetting arena objects...')
arena_data = loadmat(f'{arena_dir}{arena_items_file}', squeeze_me=True)
arena_data["perches"] = arena_data["perch_w_site"]
arena_data["feeder_perches"] = arena_data["perch_no_site"]


Loading smoothed postural keypoints and velocity...

Getting reprojection error...

Getting arena objects...


In [4]:
params = {
        "reproj_thresh": 10,  # Maximum reprojection error to count as a valid frame (pixels)
        "speed_thresh": 1/2,  # Threshold for feet 'not moving' (normalized units/second, e.g., output of Kalman filter)
        "cache_height_thresh": 0.023,  # Threshold for beak low enough for site interaction
        "merge_dur_thresh": 50,  # Threshold below which events at the same site *must* be merged (in frames at 50 fps)
        "feeder_height_thresh": 0.05,  # Threshold for beak low enough for feeder interaction
        # "feeder_radius_thresh": 1.75 / 13,  # Threshold for beak close enough to the center for feeder interaction
        "water_height_thresh": 0.06,  # Threshold for beak low enough for water interaction
        "water_radius_thresh": 0.625 / 13,  # Threshold for beak close enough to the center for water dish interaction
        "beak_foot_dist_thresh": 0.03,  # Distance threshold to count beak and feet near enough for eating
        # "cache_radius_tol": 1.005,  # Scale factor to adjust cache site locations (>1 means further from arena center) to better match the center of beak interactions
        "state_median_win": 5,  # Median window for filtering state status to prevent transient blips
    }

In [5]:
''' Calculate temporary variables '''
# beak and foot positions
beak_pos = np.mean(smooth_pts[:, [0, 1]], axis=1)  # avg beak position
foot_pos = np.mean(smooth_pts[:, [10, 14]], axis=1)  # avg foot position
beak_foot_dist = np.sqrt(np.sum((beak_pos - foot_pos)**2, axis=1))  # Euclidean dist beak to foot

# data params
n_frames = beak_pos.shape[0]  # Number of frames
n_cache_sites = len(arena_data["caches"])  # Number of cache sites
n_perches = len(arena_data["perches"])  # Number of perches
n_feeders = len(arena_data["feeder_perches"])  # Number of feeder perches

''' Utility indicators '''
valid_frames = body_reproj_error < params["reproj_thresh"]
feet_still = foot_speed < params["speed_thresh"]
beak_low_cache = beak_pos[:, 2] < params["cache_height_thresh"]
beak_low_feeder = beak_pos[:, 2] < params["feeder_height_thresh"]
beak_low_water = beak_pos[:, 2] < params["water_height_thresh"]
beak_low_feet = beak_foot_dist < params["beak_foot_dist_thresh"]
beak_radius_water = np.sqrt(np.sum(beak_pos[:, :2]**2, axis=1)) < params["water_radius_thresh"]

In [7]:
''' Detect interactions '''
print('\nDetecting perch interactions...')
# Feet on perches
feet_on_perch = np.zeros((n_frames, n_feeders), dtype=bool)

for n in range(n_feeders):
    convex_hull = arena_data["feeder_perches"][n]["ConvexHull"]
    path = Path(convex_hull)
    tmp = path.contains_points(foot_pos[:, :2])
    feet_on_perch[:, n] = tmp & feet_still & valid_frames


Detecting perch interactions...


IndexError: index 84 is out of bounds for axis 1 with size 4

In [ ]:
# Median filter interactions
state_median_win = params["state_median_win"]
print(f"\n Median state filtering with {state_median_win} frames")
feet_on_perch = medfilt(feet_on_perch.astype(float), kernel_size=(state_median_win, 1)).astype(bool)

In [ ]:
# Always merge perch interactions at the same site
new_perch, end_perch, perch_num = detect_stateChanges_selfmerge(feet_on_perch)